In [2]:
import pandas as pd

matches = pd.read_csv("matches.csv")
deliveries = pd.read_csv("deliveries.csv")
print ("matches columns:", matches.columns)
print ("deliveries columns:", deliveries.columns)

matches columns: Index(['id', 'Season', 'city', 'date', 'team1', 'team2', 'toss_winner',
       'toss_decision', 'result', 'dl_applied', 'winner', 'win_by_runs',
       'win_by_wickets', 'player_of_match', 'venue', 'umpire1', 'umpire2',
       'umpire3'],
      dtype='str')
deliveries columns: Index(['match_id', 'inning', 'batting_team', 'bowling_team', 'over', 'ball',
       'batsman', 'non_striker', 'bowler', 'is_super_over', 'wide_runs',
       'bye_runs', 'legbye_runs', 'noball_runs', 'penalty_runs',
       'batsman_runs', 'extra_runs', 'total_runs', 'player_dismissed',
       'dismissal_kind', 'fielder'],
      dtype='str')


In [3]:
first_innings = deliveries[deliveries["inning"] == 1]
total_score_df = first_innings.groupby("match_id")["total_runs"].sum().reset_index()
total_score_df["total_runs"] = total_score_df["total_runs"]+1
total_score_df.rename(columns={"total_runs": "target"}, inplace=True)
print (total_score_df.head())

   match_id  target
0         1     208
1         2     185
2         3     184
3         4     164
4         5     158


In [4]:
matches.rename(columns={"id": "match_id"}, inplace=True)

matches_with_target = matches.merge(total_score_df, on="match_id", how = 'left')
matches_with_target = matches_with_target[matches_with_target['dl_applied'] == 0]

def identify_toss_advantage(row):
    if row['toss_winner'] == row['team2']:
        return 1 
    else:
        return 0
    
matches_with_target['chaser_won_toss'] = matches_with_target.apply(identify_toss_advantage, axis = 1) 
match_context = matches_with_target[['match_id', 'city', 'winner', 'target', 'chaser_won_toss']]

In [19]:
second_innings = deliveries[deliveries['inning'] == 2]
chase_df = second_innings.merge(match_context, on = 'match_id', how = 'inner')
print(chase_df.head())

   match_id  inning                 batting_team         bowling_team  over  \
0         1       2  Royal Challengers Bangalore  Sunrisers Hyderabad     1   
1         1       2  Royal Challengers Bangalore  Sunrisers Hyderabad     1   
2         1       2  Royal Challengers Bangalore  Sunrisers Hyderabad     1   
3         1       2  Royal Challengers Bangalore  Sunrisers Hyderabad     1   
4         1       2  Royal Challengers Bangalore  Sunrisers Hyderabad     1   

   ball        batsman    non_striker   bowler  is_super_over  ...  \
0     1       CH Gayle  Mandeep Singh  A Nehra              0  ...   
1     2  Mandeep Singh       CH Gayle  A Nehra              0  ...   
2     3  Mandeep Singh       CH Gayle  A Nehra              0  ...   
3     4  Mandeep Singh       CH Gayle  A Nehra              0  ...   
4     5  Mandeep Singh       CH Gayle  A Nehra              0  ...   

   batsman_runs  extra_runs  total_runs  player_dismissed  dismissal_kind  \
0             1           0

In [20]:
chase_df['current_score'] = chase_df.groupby('match_id')['total_runs'].cumsum()
chase_df['runs_left'] = chase_df['target'] - chase_df['current_score']

In [21]:
chase_df['balls_bowled'] = (chase_df['over'] - 1) * 6 + chase_df['ball']
chase_df['balls_left'] = 120 - chase_df['balls_bowled']
chase_df['balls_left'] = chase_df['balls_left'].apply(lambda x: 0 if x < 0 else x)

In [22]:
chase_df['player_dismissed'] = chase_df['player_dismissed'].fillna('0')
chase_df['is_wicket'] = chase_df['player_dismissed'].apply(lambda x:0 if x == '0' else 1)
wickets_fallen_df = chase_df.groupby('match_id')['is_wicket'].cumsum()
chase_df['wickets_left'] = 10 - wickets_fallen_df


In [23]:
chase_df['crr'] = (chase_df['current_score'] * 6) / (chase_df['balls_bowled'] )
chase_df['rrr'] = chase_df.apply(lambda row: (row['runs_left'] * 6) / row['balls_left'] if row['balls_left'] > 0 else 0, axis = 1)


In [24]:
def check_winner(row):
    return 1 if row['batting_team'] == row['winner'] else 0
chase_df['result'] = chase_df.apply(check_winner, axis = 1)

In [25]:
sample_match = chase_df[chase_df['match_id'] == chase_df['match_id'].iloc[0]]
print(sample_match[['over', 'ball', 'current_score', 'runs_left', 'balls_left', 'wickets_left', 'crr', 'rrr', 'result']].head(10))

   over  ball  current_score  runs_left  balls_left  wickets_left        crr  \
0     1     1              1        207         119            10   6.000000   
1     1     2              1        207         118            10   3.000000   
2     1     3              1        207         117            10   2.000000   
3     1     4              3        205         116            10   4.500000   
4     1     5              7        201         115            10   8.400000   
5     1     6             11        197         114            10  11.000000   
6     2     1             11        197         113            10   9.428571   
7     2     2             11        197         112            10   8.250000   
8     2     3             12        196         111            10   8.000000   
9     2     4             12        196         110            10   7.200000   

         rrr  result  
0  10.436975       0  
1  10.525424       0  
2  10.615385       0  
3  10.603448       0  
4  1

In [26]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

features_columns = ['chaser_won_toss','runs_left', 'balls_left', 'wickets_left', 'total_runs', 'crr', 'rrr']
x = chase_df[features_columns]
y = chase_df['result']

x_train, x_test, y_train, y_test = train_test_split(x,y, test_size = 0.2, random_state = 42, stratify = y)

print('training feature shape',x_train.shape)
print('testing feature shape',x_test.shape)

training feature shape (68063, 7)
testing feature shape (17016, 7)


In [27]:
model = LogisticRegression(max_iter = 1000)
model.fit(x_train, y_train)
y_pred = model.predict(x_test)

print('initial model accuracy', accuracy_score(y_test, y_pred))
print('classification report\n', classification_report(y_test, y_pred))

initial model accuracy 0.7702750352609309
classification report
               precision    recall  f1-score   support

           0       0.76      0.74      0.75      7968
           1       0.78      0.80      0.79      9048

    accuracy                           0.77     17016
   macro avg       0.77      0.77      0.77     17016
weighted avg       0.77      0.77      0.77     17016



In [36]:
import numpy as np

hypothetical_ball = np.array([[0,30,12,7, 180, 8.5, 15.0]])
probability = model.predict_proba(hypothetical_ball)[0]

win_prob = round(probability[1]*100, 1)
loss_prob = round(probability[0]*100, 1)

print(f"Win Probability: {win_prob}%")
print(f"Loss Probability: {loss_prob}%")

Win Probability: 96.8%
Loss Probability: 3.2%


c:\Users\sudha\Desktop\ipldata\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


In [37]:
import pickle as pkl
with open('pipe.pkl', 'wb') as file:
    pkl.dump(model, file)
    print('saved the file')

saved the file


In [5]:
# Export the clean match context dataset to a new CSV for Power BI
match_context.to_csv('powerbi_ipl_clean.csv', index=False)
print("Clean data exported successfully for Power BI!")

Clean data exported successfully for Power BI!
